# SciTeX Linter — Notebook Usage

This notebook demonstrates using the scitex-linter Python API programmatically.

## 1. Lint Source Code Directly

In [1]:
from scitex_linter.checker import lint_source
from scitex_linter.formatter import format_issue

# A script with several SciTeX anti-patterns
bad_code = """\
import matplotlib.pyplot as plt
import numpy as np

def main():
    data = np.load("/tmp/data.npy")
    fig, ax = plt.subplots()
    ax.plot(data)
    plt.savefig("plot.png")
    plt.show()

main()
"""

issues = lint_source(bad_code, "example.py")
for issue in issues:
    print(format_issue(issue, "example.py", color=False))
    print()

  E example.py:1:0  STX-S002
    Missing `if __name__ == '__main__'` guard
    Add `if __name__ == '__main__': main()` at the end of the script.
If this is library code (not a script), add its directory to library_dirs:
  [tool.scitex-linter]
  library_dirs = ["src", "tests", "apps", "config", "docs"]
  Or: SCITEX_LINTER_LIBRARY_DIRS=src,tests,apps,config,docs

  W example.py:1:0  STX-I001
    import matplotlib.pyplot as plt
    Use `stx.plt` instead of importing matplotlib.pyplot directly
    Replace with `stx.plt` (or `plt` injected by @stx.session).

  W example.py:5:11  STX-IO002
        data = np.load("/tmp/data.npy")
    `np.load()` detected — use `stx.io.load()` for provenance tracking
    Replace `np.load(path)` with `stx.io.load(path)`.

  W example.py:8:4  STX-IO007
        plt.savefig("plot.png")
    `.savefig()` detected — use `stx.io.save(fig, path)` for metadata embedding
    Replace `fig.savefig(path)` with `stx.io.save(fig, path)`.

  I example.py:7:4  STX-P001
        

Error in callback <bound method AutoreloadMagics.post_execute_hook of <IPython.extensions.autoreload.AutoreloadMagics object at 0x744398d29750>> (for post_execute), with arguments args (),kwargs {}:


ImportError: scitex_notebook._magic requires scitex-clew. Install with `pip install scitex-clew`.

## 2. Lint a File on Disk

In [2]:
from scitex_linter.checker import lint_file
from scitex_linter.formatter import format_issue, format_summary

filepath = "sample_bad.py"
issues = lint_file(filepath)

for issue in issues:
    print(format_issue(issue, filepath, color=False))
    print()

print(format_summary(issues, filepath, color=False))

  E sample_bad.py:1:0  STX-S002
    Missing `if __name__ == '__main__'` guard
    Add `if __name__ == '__main__': main()` at the end of the script.
If this is library code (not a script), add its directory to library_dirs:
  [tool.scitex-linter]
  library_dirs = ["src", "tests", "apps", "config", "docs"]
  Or: SCITEX_LINTER_LIBRARY_DIRS=src,tests,apps,config,docs

  W sample_bad.py:4:0  STX-I001
    import matplotlib.pyplot as plt  # STX-I001: use stx.plt
    Use `stx.plt` instead of importing matplotlib.pyplot directly
    Replace with `stx.plt` (or `plt` injected by @stx.session).

  W sample_bad.py:15:4  STX-PA004
        os.chdir("/tmp")
    `os.chdir()` detected — scripts should be run from project root
    Remove `os.chdir()` and run scripts from the project root directory.

  W sample_bad.py:18:4  STX-IO001
        np.save("/tmp/data.npy", [1, 2, 3])
    `np.save()` detected — use `stx.io.save()` for provenance tracking
    Replace `np.save(path, arr)` with `stx.io.save(arr, pat

## 3. JSON Output

In [3]:
import json
from scitex_linter.formatter import to_json

result = to_json(issues, filepath)
print(json.dumps(result, indent=2))

{
  "file": "sample_bad.py",
  "issues": [
    {
      "rule_id": "STX-S002",
      "severity": "error",
      "category": "structure",
      "line": 1,
      "col": 0,
      "message": "Missing `if __name__ == '__main__'` guard",
      "suggestion": "Add `if __name__ == '__main__': main()` at the end of the script.\nIf this is library code (not a script), add its directory to library_dirs:\n  [tool.scitex-linter]\n  library_dirs = [\"src\", \"tests\", \"apps\", \"config\", \"docs\"]\n  Or: SCITEX_LINTER_LIBRARY_DIRS=src,tests,apps,config,docs",
      "source_line": ""
    },
    {
      "rule_id": "STX-I001",
      "severity": "warning",
      "category": "import",
      "line": 4,
      "col": 0,
      "message": "Use `stx.plt` instead of importing matplotlib.pyplot directly",
      "suggestion": "Replace with `stx.plt` (or `plt` injected by @stx.session).",
      "source_line": "import matplotlib.pyplot as plt  # STX-I001: use stx.plt"
    },
    {
      "rule_id": "STX-PA004",
    

## 4. Auto-Fix with the Fixer

In [4]:
from scitex_linter.fixer import fix_source

code_with_savefig = """\
import scitex as stx
import numpy as np

@stx.session
def main():
    data = np.load("./data.npy")
    fig, ax = stx.plt.subplots()
    ax.plot(data)
    fig.savefig("./plot.png", dpi=300)
    return 0

if __name__ == "__main__":
    main()
"""

fixed = fix_source(code_with_savefig, "example.py")
print(fixed)

import scitex as stx
import numpy as np

@stx.session
def main(
    CONFIG=stx.session.INJECTED,
    plt=stx.session.INJECTED,
    COLORS=stx.session.INJECTED,
    rngg=stx.session.INJECTED,
    logger=stx.session.INJECTED,
):
    data = stx.io.load("./data.npy")
    fig, ax = stx.plt.subplots()
    ax.plot(data)
    stx.io.save(fig, "./plot.png")
    return 0

if __name__ == "__main__":
    main()



## 5. Browse All Rules

In [5]:
from scitex_linter.rules import ALL_RULES

# Group by category
by_cat = {}
for rule in ALL_RULES.values():
    by_cat.setdefault(rule.category, []).append(rule)

for cat, rules in sorted(by_cat.items()):
    print(f"\n=== {cat.upper()} ({len(rules)} rules) ===")
    for r in sorted(rules, key=lambda x: x.id):
        print(f"  {r.id}  [{r.severity}]  {r.message}")


=== ERROR-HANDLING (1 rules) ===
  STX-EH001  [warning]  Narrow `except ImportError` may miss runtime-init errors from optional deps.

=== FIGURE (9 rules) ===
  STX-FM001  [warning]  `figsize=` detected — inch-based figure sizing is imprecise for publications
  STX-FM002  [warning]  `tight_layout()` detected — layout is unpredictable across plot types
  STX-FM003  [warning]  `bbox_inches="tight"` detected — can crop important elements unpredictably
  STX-FM004  [info]  `constrained_layout=True` detected — conflicts with mm-based layout control
  STX-FM005  [info]  `subplots_adjust()` with hardcoded fractions — fragile across figure sizes
  STX-FM006  [info]  `plt.savefig()` detected — no provenance tracking
  STX-FM007  [info]  `rcParams` direct modification detected — hard to maintain across figures
  STX-FM008  [warning]  `set_size_inches()` detected — bypasses mm-based layout control
  STX-FM009  [warning]  `ax.set_position()` detected — conflicts with mm-based layout control

===

## 6. A Clean Script (Zero Issues)

In [6]:
clean_code = """\
import scitex as stx

@stx.session
def main(
    data_path="./data.csv",
    threshold=0.5,
    CONFIG=stx.session.INJECTED,
    plt=stx.session.INJECTED,
    COLORS=stx.session.INJECTED,
    rngg=stx.session.INJECTED,
    logger=stx.session.INJECTED,
):
    df = stx.io.load(data_path)
    stx.io.save(df, "./output.csv")
    return 0

if __name__ == "__main__":
    main()
"""

issues = lint_source(clean_code, "clean.py")
print(f"Issues found: {len(issues)}")
if not issues:
    print("Zero lint issues. Fully reproducible.")

Issues found: 0
Zero lint issues. Fully reproducible.
